# 05 · Ensemble analysis: which models to combine, and how (Team Paradox)

Runs on a laptop CPU in ~2 minutes. No GPU.

**Inputs**
* `data/raw/train.csv`, `data/raw/validation.csv`
* `experiments/<run_name>/val_probs.npz`: validation probabilities from each Kaggle run (Run 1, trained on `train.csv` only, so validation is unseen)
* TF-IDF baseline: retrained here on `train.csv` only

**Questions**
1. How good is each model alone, overall and per language?
2. Does averaging models help?
3. **Language-aware ensemble:** the API does *not* send the ticket language, so we train a small language detector on the text. Does giving each language group its own blend weights beat one global weight set?

Scores for tuned blends use **nested cross-validation**: weights are tuned on 4/5 of validation and scored on the unseen 1/5 (5 folds × 3 repeats), so the numbers are honest.


In [ ]:
import itertools, json, sys
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.training.train_baseline import fit, load  # same data loading and TF-IDF as the shipped baseline

tr, va = load("train"), load("validation")
CATS = sorted(tr.category.unique())
y = np.array([CATS.index(c) for c in va.category])
lang = va.language.values
print(len(tr), len(va))


## 1. Load every model's validation probabilities

In [ ]:
P = {}
tf_model = fit(tr)
assert list(tf_model["cat_clf"].classes_) == CATS
P["tfidf"] = tf_model["cat_clf"].predict_proba(tf_model["vec"].transform(va.x))

for f in sorted((ROOT / "experiments").glob("*/val_probs.npz")):
    with np.load(f) as z:
        cat = z["cat"]
    assert cat.shape == (len(va), len(CATS)), f"{f}: unexpected shape {cat.shape}"
    P[f.parent.name] = cat
print("models:", list(P))


## 2. Each model alone
`cold_singlish` and `gedara_singlish` are the two Singlish phrasings that never appear in Singlish training tickets (cross-lingual probes).

In [ ]:
def ece(p, bins=10):
    conf, ok, e = p.max(1), p.argmax(1) == y, 0.0
    for lo in np.linspace(0, 1, bins, endpoint=False):
        m = (conf > lo) & (conf <= lo + 1 / bins)
        if m.any():
            e += m.mean() * abs(conf[m].mean() - ok[m].mean())
    return e

cold = va.text.str.contains("seethala", case=False).values
gedara = va.text.str.contains("gedarata awilla", case=False).values

def scores(p):
    pred = p.argmax(1); ok = pred == y
    row = {"acc": ok.mean(), "macro_f1": f1_score(y, pred, average="macro"), "ece": ece(p)}
    row.update({l: ok[lang == l].mean() for l in ["en", "si", "ta", "mixed", "singlish", "tanglish"]})
    row.update({"cold_singlish": ok[cold].mean(), "gedara_singlish": ok[gedara].mean()})
    return row

single = pd.DataFrame({k: scores(v) for k, v in P.items()}).T.round(3)
single


## 3. Simple averages (equal weights, nothing tuned)

In [ ]:
rows = {}
names = list(P)
for r in range(2, len(names) + 1):
    for combo in itertools.combinations(names, r):
        rows[" + ".join(combo)] = scores(sum(P[n] for n in combo) / r)
pd.DataFrame(rows).T.round(3).sort_values("acc", ascending=False)


## 4. Language detector (text only)
The API does not send `language`, so the server must detect it. Character n-grams separate scripts and romanised spellings well.

In [ ]:
lang_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 3), min_df=2, sublinear_tf=True)
lang_clf = LogisticRegression(max_iter=3000, C=10).fit(lang_vec.fit_transform(tr.x), tr.language)
lang_pred = lang_clf.predict(lang_vec.transform(va.x))
print("language detector accuracy on validation:", round((lang_pred == lang).mean(), 4))
pd.crosstab(pd.Series(lang, name="true"), pd.Series(lang_pred, name="predicted"))


## 5. Tuned blends: one global weight set vs. per language group

Groups (from the **detected** language): `native` = si, ta, mixed · `english` = en · `roman` = singlish, tanglish.
Groups instead of six languages because Tanglish and mixed have only 20 validation tickets each.

In [ ]:
GROUPS = {"native": ["si", "ta", "mixed"], "english": ["en"], "roman": ["singlish", "tanglish"]}
group = np.array([next(g for g, ls in GROUPS.items() if l in ls) for l in lang_pred])
names = list(P)
step = 0.1
grid = [w for w in itertools.product(np.arange(0, 1 + 1e-9, step), repeat=len(names)) if abs(sum(w) - 1) < 1e-6]
print(len(names), "models,", len(grid), "weight combinations")

def blend(w, idx):
    return sum(wi * P[n][idx] for wi, n in zip(w, names))

def objective(w, idx):
    pred = blend(w, idx).argmax(1)
    return (pred == y[idx]).mean() + 0.5 * f1_score(y[idx], pred, average="macro")

def best_weights(idx):
    return max(grid, key=lambda w: objective(w, idx))

nested = {"global": [], "per_group": []}
for rep in range(3):
    for trn, tst in StratifiedKFold(5, shuffle=True, random_state=rep).split(va, va.category):
        w_all = best_weights(trn)
        nested["global"].append((blend(w_all, tst).argmax(1) == y[tst]).mean())
        pred = np.empty(len(tst), dtype=int)
        for g in GROUPS:
            t_idx, s_mask = trn[group[trn] == g], group[tst] == g
            w_g = best_weights(t_idx) if len(t_idx) > 20 else w_all
            pred[s_mask] = blend(w_g, tst[s_mask]).argmax(1)
        nested["per_group"].append((pred == y[tst]).mean())

for k, v in nested.items():
    print(f"{k:10s} nested-CV accuracy: {np.mean(v):.4f} ± {np.std(v):.4f}")


## 6. Final weights (tuned on all of validation) and save them

In [ ]:
all_idx = np.arange(len(y))
final = {"models": names, "global": dict(zip(names, map(float, best_weights(all_idx))))}
for g in GROUPS:
    final[g] = dict(zip(names, map(float, best_weights(np.where(group == g)[0]))))
final["nested_cv"] = {k: [round(float(np.mean(v)), 4), round(float(np.std(v)), 4)] for k, v in nested.items()}
(ROOT / "experiments" / "ensemble_weights.json").write_text(json.dumps(final, indent=2))
pd.DataFrame({k: v for k, v in final.items() if isinstance(v, dict) and k != "nested_cv"}).round(2)


## 7. Which combination should we ship? (accuracy vs. server cost)

Every subset of encoders (TF-IDF always included, it is 3 MB), scored with the same nested CV.
`size_mb` is the int8 ONNX size on the server (LaBSE cannot be quantized, so it counts at fp32 size).

In [ ]:
SIZE_MB = {"tfidf": 3}
for n in P:
    SIZE_MB.setdefault(n, 1800 if n.startswith("labse") else 561 if n.startswith("xlmr") else 278)

def nested_scores(sub):
    g = [w for w in itertools.product(np.arange(0, 1 + 1e-9, step), repeat=len(sub)) if abs(sum(w) - 1) < 1e-6]
    bl = lambda w, idx: sum(wi * P[n][idx] for wi, n in zip(w, sub))
    ob = lambda w, idx: (bl(w, idx).argmax(1) == y[idx]).mean() + 0.5 * f1_score(y[idx], bl(w, idx).argmax(1), average="macro")
    bw = lambda idx: max(g, key=lambda w: ob(w, idx))
    out = {"global": [], "per_group": []}
    for rep in range(3):
        for trn, tst in StratifiedKFold(5, shuffle=True, random_state=rep).split(va, va.category):
            w = bw(trn); out["global"].append((bl(w, tst).argmax(1) == y[tst]).mean())
            pred = np.empty(len(tst), dtype=int)
            for gname in GROUPS:
                ti, sm = trn[group[trn] == gname], group[tst] == gname
                pred[sm] = bl(bw(ti) if len(ti) > 20 else w, tst[sm]).argmax(1)
            out["per_group"].append((pred == y[tst]).mean())
    return {k: round(float(np.mean(v)), 4) for k, v in out.items()} | {"per_group_sd": round(float(np.std(out["per_group"])), 4)}

encoders = [n for n in P if n != "tfidf"]
rows = []
for r in range(1, len(encoders) + 1):
    for sub in itertools.combinations(encoders, r):
        sub = ("tfidf",) + sub
        rows.append({"models": " + ".join(sub), "size_mb": sum(SIZE_MB[n] for n in sub), **nested_scores(sub)})
choice = pd.DataFrame(rows).sort_values("per_group", ascending=False)
choice.to_csv(ROOT / "experiments" / "ensemble_choices.csv", index=False)
choice


## Findings (fill in after running)

- 
- 
- 

**Caveats.** TF-IDF is strong on Singlish because many validation Singlish sentences closely match training ones; on unseen phrasings it can fail, so a lower TF-IDF weight for the roman group may be safer for the holdout. Blend weights come from Run 1 models (trained on train only); the shipped models are retrained on train + validation.
